# 🌊 PINNs-MVP: Kolmogorov Flow 實驗指南
## Physics-Informed Neural Networks with RANS Prior for 2D Turbulent Flow

---

**最後更新**: 2025-12-13  
**版本**: v4.0 (RANS Prior Edition)  
**新功能**: ✅ RANS Prior Loss | ✅ RANS QR Sensor | ✅ 簡化流程

---

### 📚 快速導航

1. **環境設定** - Colab 初始化與 GPU 檢查
2. **RANS 數據準備** - 生成低保真場與感測點
3. **模型訓練** - 使用 RANS Prior 訓練 PINNs
4. **結果評估** - 視覺化與物理驗證

---

### 🎯 核心特色

- **RANS 先驗引導**: 使用 k-ε RANS 場作為軟約束，改善壓力重建
- **智能感測器**: 從 RANS 場生成 QR-Pivot 最優感測點
- **完整物理**: Fourier Features + SIREN + 自適應權重 + 因果訓練
- **快速驗證**: 1000 epochs 即可收斂（2-3 小時 GPU）

---

### ⚙️ Google Colab 推薦配置

| GPU 類型 | 訓練時間 (1000 epochs) | 訂閱 |
|---------|----------------------|------|
| **NVIDIA A100** | 1-2 小時 ⭐ | Colab Pro+ |
| **NVIDIA V100** | 2-3 小時 | Colab Pro |
| **NVIDIA T4** | 3-4 小時 | 免費/Pro |

---

## Part 0: Google Colab 初始化

**⚠️ 重要**：本 Notebook 專為 **Google Colab (T4/A100 GPU)** 設計。

In [ ]:
# 0.1 檢測環境
try:
    import google.colab
    IN_COLAB = True
    print("✅ Google Colab 環境")
except ImportError:
    IN_COLAB = False
    print("⚠️  本地環境，請使用: python scripts/train/train.py --cfg <config.yml>")

In [ ]:
# 0.2 掛載 Google Drive
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    import os
    PROJECT_PATH = '/content/drive/MyDrive/pinns-mvp'
    
    if os.path.exists(PROJECT_PATH):
        os.chdir(PROJECT_PATH)
        print(f"✅ 專案目錄: {os.getcwd()}")
    else:
        raise FileNotFoundError(f"專案不存在: {PROJECT_PATH}\n請上傳專案至 Google Drive")

In [ ]:
# 0.3 檢查 GPU
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"記憶體: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = 'cuda'
else:
    print("⚠️ 僅 CPU，訓練將很慢")
    device = 'cpu'

---

## Part 1: RANS 數據準備

### 1.1 檢查 RANS 低保真場

In [ ]:
# 1.1 檢查 RANS 數據
import h5py
import numpy as np

rans_file = 'data/lowfi/kolmogorov_rans/rans_re50_kf4.h5'

with h5py.File(rans_file, 'r') as f:
    print("📊 RANS 數據結構:")
    print(f"  群組: {list(f.keys())}")
    print(f"\n  mean_field 包含: {list(f['mean_field'].keys())}")
    
    u = f['mean_field']['u'][:]
    v = f['mean_field']['v'][:]
    k = f['mean_field']['k'][:]  # 湍流動能
    nu_t = f['mean_field']['nu_t'][:]  # 渦黏度
    
    print(f"\n  網格大小: {u.shape}")
    print(f"  u 範圍: [{u.min():.4f}, {u.max():.4f}]")
    print(f"  v 範圍: [{v.min():.4f}, {v.max():.4f}]")
    print(f"  k 範圍: [{k.min():.4f}, {k.max():.4f}]")

### 1.2 視覺化 RANS 場

In [ ]:
# 1.2 繪製 RANS 場
import matplotlib.pyplot as plt

with h5py.File(rans_file, 'r') as f:
    u = f['mean_field']['u'][:]
    v = f['mean_field']['v'][:]
    X = f['mean_field']['X'][:]
    Y = f['mean_field']['Y'][:]

speed = np.sqrt(u**2 + v**2)
vorticity = np.gradient(v, axis=1) - np.gradient(u, axis=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

im1 = ax1.contourf(X, Y, speed, levels=50, cmap='viridis')
ax1.set_title('RANS Speed Field', fontsize=14, fontweight='bold')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
plt.colorbar(im1, ax=ax1, label='|u|')

im2 = ax2.contourf(X, Y, vorticity, levels=50, cmap='RdBu_r')
ax2.set_title('RANS Vorticity', fontsize=14, fontweight='bold')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
plt.colorbar(im2, ax=ax2, label='ω')

plt.tight_layout()
plt.savefig('results/rans_field_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ RANS 場視覺化完成")

### 1.3 檢查 RANS QR Sensor

In [ ]:
# 1.3 檢查 RANS sensor
sensor_file = 'data/lowfi/kolmogorov_rans/sensors_K100_rans.npz'

sensors = np.load(sensor_file)

print("📍 RANS Sensor 配置:")
print(f"  感測點數: {sensors['K']}")
print(f"  方法: {sensors['method']}")
print(f"  來源: {sensors['source']}")
print(f"  X 範圍: [{sensors['sensor_x'].min():.4f}, {sensors['sensor_x'].max():.4f}]")
print(f"  Y 範圍: [{sensors['sensor_y'].min():.4f}, {sensors['sensor_y'].max():.4f}]")

metrics = sensors['metrics'].item()
print(f"\n  條件數: {metrics.get('condition_number', 'N/A'):.2e}")

if metrics.get('condition_number', 1000) < 100:
    print("  ✅ 優秀（< 100）")
elif metrics.get('condition_number', 1000) < 500:
    print("  ✅ 良好（100-500）")
else:
    print("  ⚠️ 可接受（> 500）")

### 1.4 視覺化感測點分佈

In [ ]:
# 1.4 繪製感測點疊加在 RANS 場上
fig, ax = plt.subplots(figsize=(10, 10))

# RANS 背景
with h5py.File(rans_file, 'r') as f:
    u = f['mean_field']['u'][:]
    v = f['mean_field']['v'][:]
    X = f['mean_field']['X'][:]
    Y = f['mean_field']['Y'][:]

speed = np.sqrt(u**2 + v**2)
im = ax.contourf(X, Y, speed, levels=50, cmap='viridis', alpha=0.8)

# 感測點
ax.scatter(sensors['sensor_x'], sensors['sensor_y'],
           c='red', s=30, marker='x', linewidths=2,
           label=f"QR Sensors (K={sensors['K']})")

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('RANS QR-Pivot Sensors Distribution', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.set_aspect('equal')
plt.colorbar(im, ax=ax, label='Speed |u|')

plt.tight_layout()
plt.savefig('results/rans_sensor_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ RANS Sensor 視覺化完成")

---

## Part 2: 模型訓練

### 2.1 檢查配置文件

In [ ]:
# 2.1 檢查配置文件
import yaml

config_file = 'configs/kolmogorov_re50_kf4_K100_rans_prior_1k.yml'

with open(config_file, 'r') as f:
    config = yaml.safe_load(f)

print("📋 訓練配置摘要:")
print(f"  實驗名稱: {config['experiment']['name']}")
print(f"  版本: {config['experiment']['version']}")
print(f"\n🔧 模型:")
print(f"  類型: {config['model']['type']}")
print(f"  寬度: {config['model']['width']}")
print(f"  深度: {config['model']['depth']}")
print(f"  Fourier: {config['model']['fourier_features']['enabled']}")
print(f"\n🎯 訓練:")
print(f"  Epochs: {config['training']['epochs']}")
print(f"  優化器: {config['training']['optimizer']['type']}")
print(f"  學習率: {config['training']['optimizer']['lr']}")
print(f"\n🌊 RANS Prior:")
print(f"  啟用: {config['lowfi_prior']['enabled']}")
print(f"  權重: {config['lowfi_prior']['consistency_weight']}")
print(f"  數據: {config['lowfi_prior']['data_path']}")
print(f"\n📍 感測器:")
print(f"  K: {config['sensors']['K']}")
print(f"  方法: {config['sensors']['selection_method']}")

### 2.2 啟動訓練（GPU 加速）

In [ ]:
# 2.2 訓練模型（使用 RANS Prior）
# 訓練時間：T4 ~3-4 小時 | A100 ~1-2 小時（1000 epochs）

!python scripts/train/train.py \
  --cfg configs/kolmogorov_re50_kf4_K100_rans_prior_1k.yml \
  --device {device}

# 檢查點保存至: checkpoints/kolmogorov_re50_kf4_K100_rans_prior/
# 結果保存至: results/kolmogorov_re50_kf4_K100_rans_prior/

### 2.3 監控訓練進度

In [ ]:
# 2.3 檢查訓練日誌
import os

log_dir = 'log'
log_files = [f for f in os.listdir(log_dir) if 'kolmogorov_re50' in f]

if log_files:
    latest_log = sorted(log_files)[-1]
    print(f"📄 最新日誌: {latest_log}\n")
    !tail -30 log/{latest_log}
else:
    print("⚠️ 未找到訓練日誌")

In [ ]:
# 2.4 列出檢查點
checkpoint_dir = 'checkpoints/kolmogorov_re50_kf4_K100_rans_prior'

if os.path.exists(checkpoint_dir):
    checkpoints = sorted(os.listdir(checkpoint_dir))
    print(f"📂 檢查點 ({len(checkpoints)} 個):\n")
    for ckpt in checkpoints[-5:]:  # 顯示最近 5 個
        path = os.path.join(checkpoint_dir, ckpt)
        size = os.path.getsize(path) / 1e6
        print(f"  {ckpt} ({size:.1f} MB)")
else:
    print("⚠️ 檢查點目錄不存在")

---

## Part 3: 結果評估

### 3.1 評估最佳模型

In [ ]:
# 3.1 使用統一評估腳本
!python scripts/evaluate/evaluate_checkpoint.py \
  --checkpoint checkpoints/kolmogorov_re50_kf4_K100_rans_prior/best_model.pth \
  --config configs/kolmogorov_re50_kf4_K100_rans_prior_1k.yml \
  --output results/evaluation_rans_prior/

# 生成指標：
# - 相對 L2 誤差 (u, v, p)
# - 物理殘差 (連續性、動量)
# - 壓力梯度誤差 (dpdx, dpdy)

### 3.2 視覺化結果

In [ ]:
# 3.2 生成完整視覺化
!python scripts/visualize/visualize_results.py \
  --checkpoint checkpoints/kolmogorov_re50_kf4_K100_rans_prior/best_model.pth \
  --reference data/kolmogorov_dns/dns_re50_t100.h5 \
  --output results/evaluation_rans_prior/visualizations/

# 生成圖表：
# - field_comparison.png: 預測 vs 真值 vs 誤差
# - energy_spectrum.png: 能譜對比
# - statistics.png: 統計量分析

In [ ]:
# 3.3 顯示結果圖
from IPython.display import Image, display

viz_dir = 'results/evaluation_rans_prior/visualizations'

if os.path.exists(viz_dir):
    print("📊 場重建對比:")
    display(Image(filename=f'{viz_dir}/field_comparison.png', width=1200))
    
    print("\n📈 能譜對比:")
    display(Image(filename=f'{viz_dir}/energy_spectrum.png', width=800))
else:
    print("⚠️ 視覺化結果尚未生成")

### 3.3 量化評估總結

In [ ]:
# 3.4 讀取評估指標
import json

metrics_file = 'results/evaluation_rans_prior/metrics.json'

if os.path.exists(metrics_file):
    with open(metrics_file, 'r') as f:
        metrics = json.load(f)
    
    print("="*70)
    print("🏆 評估總結")
    print("="*70)
    
    print("\n📊 場重建誤差:")
    u_l2 = metrics['field_errors']['u_l2_error'] * 100
    v_l2 = metrics['field_errors']['v_l2_error'] * 100
    p_l2 = metrics['field_errors']['p_l2_error'] * 100
    
    print(f"  u: {u_l2:.2f}% {'✅' if u_l2 < 15 else '❌'} (目標 < 15%)")
    print(f"  v: {v_l2:.2f}% {'✅' if v_l2 < 15 else '❌'} (目標 < 15%)")
    print(f"  p: {p_l2:.2f}% {'✅' if p_l2 < 20 else '❌'} (目標 < 20%)")
    
    if 'pressure_gradient' in metrics:
        print("\n📐 壓力梯度誤差 (RANS Prior 改善):")
        dpdx = metrics['pressure_gradient']['dpdx_l2'] * 100
        dpdy = metrics['pressure_gradient']['dpdy_l2'] * 100
        print(f"  ∂p/∂x: {dpdx:.2f}% {'✅' if dpdx < 30 else '❌'} (目標 < 30%)")
        print(f"  ∂p/∂y: {dpdy:.2f}% {'✅' if dpdy < 30 else '❌'} (目標 < 30%)")
    
    print("\n⚖️ 物理守恆:")
    div = metrics['physics']['divergence_error']
    print(f"  連續性: {div:.2e} {'✅' if div < 1e-3 else '❌'} (目標 < 1e-3)")
    
    print("\n" + "="*70)
else:
    print("⚠️ 評估指標文件不存在")

---

## 📚 參考資料

### 📖 文檔
- [`docs/TECHNICAL_DOCUMENTATION.md`](docs/TECHNICAL_DOCUMENTATION.md) - 完整技術文檔
- [`configs/templates/README.md`](configs/templates/README.md) - 配置模板指南
- [`scripts/README.md`](scripts/README.md) - 腳本使用說明

### 🛠️ 關鍵腳本
- `scripts/train/train.py` - 主訓練器 (新位置)
- `scripts/evaluate/evaluate_checkpoint.py` - 檢查點評估
- `scripts/visualize/visualize_results.py` - 結果視覺化
- `scripts/generate/sensors/generate_sensors_periodic_qr.py` - QR-Pivot 感測器

### 📊 配置文件
- `configs/kolmogorov_re50_kf4_K100_rans_prior_1k.yml` - RANS Prior 訓練配置
- `configs/templates/2d_quick_baseline.yml` - 快速基線模板

### 📝 文獻
- Musacchio & Boffetta (2014) - Kolmogorov Flow Reynolds 數定義
- Raissi et al. (2019) - Physics-Informed Neural Networks
- Wang et al. (2021) - VS-PINN 變數縮放

---

**專案倉庫**: https://github.com/latteine1217/pinns-mvp  
**授權**: MIT License